# SNN Grad-CAM: Step-by-step walkthrough

This notebook validates per-timestep Grad-CAM for SpikingVGG11:

- Load config, data, model
- Choose a target conv layer
- Generate Grad-CAMs per time step
- Visualize activations/gradients sanity checks
- Overlay CAMs on frames and plot time series

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from src.snn_gradcam import SNNGradCAM, get_default_conv_layer
from src.utils.config_utils import setup_config_dictionary
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cam_time_series, combine_polarities

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(torch.seed())
torch.manual_seed(2329669718442129658)
# torch.manual_seed(7592244650879848172)
config = setup_config_dictionary()
config["DATASET_NAME"] = "tiny_imagenet"
config["MODEL_NAME"] = "vgg11"
config["NORM_TYPE"] = "bntt"
print('Device:', device)
print('Dataset:', config['DATASET_NAME'])
print('Model:', config['MODEL_NAME'])

In [ ]:
train_loader, test_loader = get_dataloaders(
    config['DATASET_NAME'],
    batch_size=1,
)

events, labels = next(iter(test_loader))  # (B,T,C,H,W) or (B,C,H,W)
if events.ndim == 4:
    events = events.unsqueeze(1)
B, T, C, H, W = events.shape
print('Sample shape:', (B, T, C, H, W))

model = get_model(
    config['MODEL_NAME'],
    num_classes=len(train_loader.dataset.classes),
    num_channels=C,
    time_steps=config['N_TIME_BINS'],
    beta=config['BETA'],
    norm_type=config['NORM_TYPE'],
).to(device)

interactive_load_model(model, config['MODEL_SAVE_DIR'], config['DATASET_NAME'])
model.eval()

events = events.to(device)
labels = labels.to(device)
print('Label (gt):', int(labels[0].item()))

## Pick a target convolutional layer

We default to the deepest Conv2d layer.

In [ ]:
conv_layers = [m for m in model.modules() if isinstance(m, torch.nn.Conv2d)]
print('Total Conv2d layers:', len(conv_layers))

# Heuristic: take a deeper layer if available, else last
layer_index = min(6, len(conv_layers) - 1)
target_layer = conv_layers[layer_index]
print('Using conv layer index:', layer_index, '->', target_layer)

print('Target layer:', target_layer)
gradcam = SNNGradCAM(model, target_layer, relu=True)

## Forward and prediction

We use time-mean score for class selection and gradient backprop.

In [ ]:
outputs = model(events)  # (B, T, num_classes)
time_mean = outputs.mean(dim=1)
pred = time_mean.argmax(dim=1)
print('Predicted class:', int(pred[0].item()))
print('Time-mean confidence:', float(time_mean[0, pred[0]].softmax(dim=0).max().item()))

## Generate per-timestep Grad-CAMs

We aggregate class scores across time with mean, backprop through the SNN, and collect activations/gradients at each time step.

In [ ]:
cams = gradcam.generate(
    events,
    score_aggregation='mean',
    normalize_per_timestep=False,
    upsample_to_input=True,
    class_idx=
    #int(labels[0].item())
    int(pred[0].item()),
)  # (B, T, H, W)
print('CAMs shape:', tuple(cams.shape))

## Visualize

- If input has 2 polarities, combine to grayscale for overlays.
- Plot time series of global CAM intensity with selected overlays.

In [ ]:
frames = events[0].detach().cpu()  # (T, C, H, W)
if frames.shape[1] == 2:
    fr_np = frames.permute(0, 2, 3, 1).numpy()  # (T,H,W,2)
    comb = combine_polarities(fr_np)  # (T,H,W)
    frames_vis = torch.from_numpy(comb).unsqueeze(1)  # (T,1,H,W)
else:
    frames_vis = frames  # assume RGB

stride = max(1, frames_vis.shape[0] // 12)
cts = plot_cam_time_series(
    frames_vis,
    cams[0],
    aggregate='mean',
    frame_stride=stride,
    per_frame_cam_norm=True,
)

## Sanity checks

Visualize a few CAM frames and confirm spatial support and dynamics.

In [ ]:
Ttot = cams.shape[1]
sel_ts = np.linspace(0, Ttot-1, min(6, Ttot)).astype(int).tolist()
plt.figure(figsize=(3*len(sel_ts), 3))
for i, t in enumerate(sel_ts):
    plt.subplot(1, len(sel_ts), i+1)
    plt.imshow(cams[0, t].detach().cpu(), cmap='jet')
    plt.title(f't={t}')
    plt.axis('off')
plt.tight_layout(); plt.show()

### Notes

- Change `score_aggregation` to `'sum'` if desired.
- Set `normalize_per_timestep=True` to scale each CAM to [0,1] individually.
- Swap `target_layer` to earlier convs to compare spatial support.